## Agentic AI pattern: LLM as a judge ##

In [1]:
import os
import sys
import json
import requests
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from pypdf import PdfReader
import gradio as gr
from IPython.display import Markdown, display

%load_ext autoreload
%autoreload 2
import agentix

print(f'Package version: {agentix.__version__}')
print(f'Authors:         {agentix.__authors__}')
print(f'Python version:  {sys.version}')

Package version: 0.0.1
Authors:         Andreas Werdich
The Core for Computational Biomedicine at Harvard Medical School
https://dbmi.hms.harvard.edu/about-dbmi/core-computational-biomedicine
Python version:  3.12.13 (main, Jul 23 2026, 14:43:28) [Clang 22.1.3 ]


### MODELS ###

In [2]:
# Azure OpenAI
load_dotenv()
api_key = os.environ['AZURE_API_KEY']
base_url = f'https://azure-ai.hms.edu/openai/v1'
openai = OpenAI(base_url=base_url, api_key=api_key, default_headers={'api-key': api_key})

In [3]:
# Ollama
ollama_url = 'http://localhost:11434'
OLLAMA_BASE_URL = f'{ollama_url}/v1'
print(requests.get(ollama_url).content)
models = requests.get(f'{ollama_url}/v1/models').json()
for model in models.get('data'):
    print(model.get('id'))
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

ConnectionError: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: / (Caused by NewConnectionError("HTTPConnection(host='localhost', port=11434): Failed to establish a new connection: [Errno 111] Connection refused"))

In [4]:
# Data directory
data_dir = Path(os.environ['HOME']) / 'gitrepos' / 'agentix' / 'data'
print(data_dir)

/home/andreas/gitrepos/agentix/data


In [10]:
# Load the pdf file and turn it into text
reader = PdfReader(data_dir / 'linkedin.pdf')
linkedin = ''
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

# Read the summary
with open(data_dir / 'summary.txt', 'r', encoding='utf-8') as f:
    summary = f.read()

In [12]:
print(summary)

Andreas Augustinus Werdich is a physicist, biomedical engineer, and data scientist serving as a Research Scientist in Biomedical Informatics at Harvard Medical School. His research has focused on cardiac electrophysiology, heart development, imaging, and the mechanisms underlying arrhythmias. He has published 36 peer-reviewed papers, including work in Nature journals, and has developed software and computational methods for biomedical research. As Director of AI and Data Science at HMS’s Core for Computational Biomedicine, he leads projects applying artificial intelligence and computer vision to medical education, dentistry, cardiovascular research, and clinical data analysis.


We have a profile (CV) and a summary. Let's create a system prompt.

In [1]:
system_prompt = f'''

# Your role

You are a digital twin running on a website, chatting with visitors of the website.
You represent the person who's website you are on.
You answer questions related to their career, background, skills and experience.

Here are the details of the person you are representing:

{summary}

If asked, you explain clearly that you are an AI that is the digital twin of this person.

# Context

Here is a summary of the person's profile so that you can answer questions:

{linkedin}

# Rules

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website. 
Avoid answering questions that are not related to the user's career, background, skills and experience;
steer the conversation back to professional topics.and

Always stay in character as the digital twin of the person you are representing. Represent the person.

IMPORTANT: If you do not know the answer, say so. Never make up an answer.
If the user asks about something that is not in the context, say that you do not know.

'''

NameError: name 'summary' is not defined